In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

# ==============================================================================
# 0. DOSSIERS DE SORTIE
# ==============================================================================
output_dir = "resultats_Wenner_Schlumberger"
fig_dir = os.path.join(output_dir, "figures")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

def save_figure(fig, filename):
    """Sauvegarde systématique de chaque figure."""
    path = os.path.join(fig_dir, filename)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[OK] Figure enregistrée : {path}")
# ==============================================================================
# 1. GEOMETRIE DU MODELE SYNTHETIQUE GEOLOGIQUE
# ==============================================================================
n_electrodes = 36
spacing = 5

x_max = (n_electrodes - 1) * spacing
z_bottom = 80
# -----------------------------
# Topographie
# -----------------------------
x_topo = np.linspace(0, x_max, n_electrodes)

y_topo = np.piecewise(
    x_topo,
    [
        x_topo < 20,
        (x_topo >= 20) & (x_topo < 120),
        x_topo >= 120],
    [
        lambda x: 150 + 0.5 * x,
        lambda x: 160,
        lambda x: 160 - 0.1 * (x - 120)])

points_topo = list(zip(x_topo, y_topo))
# ==============================================================================
# 2. DOMAINE GEOLOGIQUE
# ==============================================================================
poly_points = points_topo + [
    [x_max, z_bottom],
    [0, z_bottom]]
world = mt.createPolygon(
    poly_points,
    isClosed=True,
    marker=1)
for p in points_topo:
    world.createNode(p)

# ==============================================================================
# 3. SOCLE ROCHEUX
# ==============================================================================
points_socle = [
    [0, z_bottom],
    [0, 110],
    [50, 120],
    [75, 125],
    [90, 130],
    [115, 135],
    [x_max, 153],
    [x_max, z_bottom]]

socle = mt.createPolygon(
    points_socle,
    isClosed=True,
    marker=2)
# ==============================================================================
# 4. ANOMALIE CONDUCTRICE (DMA / DEPOT MINIER)
# ==============================================================================

anomaly = mt.createCircle(
    pos=[75, 145],
    radius=[50, 8],
    marker=3,
    area=1)
# ==============================================================================
# 5. GEOMETRIE COMPLETE
# ==============================================================================
geometry = world + socle + anomaly
# ==============================================================================
# 6. AFFICHAGE + SAUVEGARDE DU MODELE GEOLOGIQUE
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 5))
pg.show(
    geometry,
    ax=ax,
    markers=True)
ax.set_title("Modèle géologique synthétique Wenner-Schlumberger")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Altitude / profondeur (m)")
plt.tight_layout()
save_figure(fig, "01_modele_geologique.png")
plt.show()
plt.close(fig)
# ==============================================================================
# 7. MAILLAGE ELEMENTS FINIS (FEM)
# ==============================================================================
mesh = mt.createMesh(
    geometry,
    quality=34,
    area=2,
    smooth=True)
fig, ax = plt.subplots(figsize=(10, 5))
pg.show(
    mesh,
    ax=ax,
    markers=True,
    showMesh=True)
ax.set_title("Maillage FEM du modèle synthétique")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Altitude / profondeur (m)")
plt.tight_layout()
save_figure(fig, "02_maillage_FEM.png")
plt.show()
plt.close(fig)
# ==============================================================================
# 8. DISPOSITIF ELECTRIQUE WENNER-SCHLUMBERGER
#    IMPORTANT : "ws" correspond à Wenner-Schlumberger
# ==============================================================================
scheme = ert.createData(
    elecs=points_topo,
    schemeName="slm",
    nMax=6)
print("--------------------------------")
print("Dispositif : Wenner-Schlumberger")
print("Nombre de mesures :", scheme.size())
print("--------------------------------")
# ==============================================================================
# 9. MODELE DE RESISTIVITE VRAI
# ==============================================================================
rhomap = [
    [0, 10],     # couverture / résidus conducteurs
    [1, 50],     # encaissant
    [2, 150],    # socle résistant
    [3, 2]       # anomalie DMA très conductrice]
# ==============================================================================
# 10. SIMULATION SYNTHETIQUE AVEC BRUIT
# ==============================================================================

data = ert.simulate(
    mesh,
    scheme=scheme,
    res=rhomap,
    noiseLevel=0.01,
    noiseAbs=1e-6,
    seed=2025)
# Nettoyage des données invalides
data.remove(data["rhoa"] <= 0)
print(data)
# ==============================================================================
# 11. PSEUDOSECTION DES DONNEES SYNTHETIQUES
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 5))
ert.show(data, ax=ax)
ax.set_title("Pseudosection ")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Altitude / profondeur apparente")
plt.tight_layout()
save_figure(fig, "03_pseudosection_synthetique.png")
plt.show()
plt.close(fig)

# ==============================================================================
# 12. INVERSION ERT
# ==============================================================================
mgr = ert.ERTManager(
    data,
    sr=True,
    verbose=True)
model = mgr.invert(
    lam=20,
    maxIter=10,
    paraMaxCellSize=1.5,
    zWeight=0.5,
    robustData=True,
    growthRate=1.1)
rms = mgr.inv.relrms()
print("==============================")
print(" RMS FINAL =", rms, "%")
print("==============================")
# ==============================================================================
# 13. RESULTAT D'INVERSION
# ==============================================================================
fig, ax = plt.subplots(figsize=(12, 6))
mgr.showResult(
    ax=ax,
    cMin=3,
    cMax=150,
    logScale=True,
    cMap="jet")
ax.set_ylim(80, 165)
ax.set_title(f"Inversion Wenner-Schlumberger - RMS={rms:.2f}%")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Profondeur (m)")
plt.tight_layout()
save_figure(fig, "04_resultat_inversion.png")
plt.show()
plt.close(fig)
# ==============================================================================
# 14. SCHEMA FINAL DU FLUX DE TRAVAIL
#    (le schéma demandé apparaît à la fin)
# ==============================================================================

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")

boxes = [
    (0.05, 0.35, "1. Géométrie\n+ topographie"),
    (0.24, 0.35, "2. Maillage FEM"),
    (0.43, 0.35, "3. Dispositif\nWenner-Schlumberger"),
    (0.62, 0.35, "4. Simulation\n+ bruit"),
    (0.80, 0.35, "5. Inversion\nERT"),]

# Dessin des boîtes
for x, y, txt in boxes:
    ax.text(
        x, y, txt,
        ha="center", va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black", linewidth=1.2) )

# Flèches
arrow_y = 0.35
for x1, x2 in [(0.12, 0.19), (0.31, 0.38), (0.50, 0.57), (0.69, 0.76)]:
    ax.annotate(
        "",
        xy=(x2, arrow_y),
        xytext=(x1, arrow_y),
        arrowprops=dict(arrowstyle="->", lw=1.8) )

ax.text(
    0.5, 0.08,
    "Sorties enregistrées : figures, modèle inversé, maillage, données synthétiques et dossier de résultats",
    ha="center", va="center", fontsize=10)

ax.set_title("Schéma global du traitement ERT synthétique", fontsize=13, pad=15)
plt.tight_layout()
save_figure(fig, "05_schema_global.png")
plt.show()
plt.close(fig)
# ==============================================================================
# 15. SAUVEGARDE DES RESULTATS NUMERIQUES
#    (mise à la fin, comme demandé)
# ==============================================================================

# Modèle inversé
np.save(
    os.path.join(output_dir, "modele_inverse_Wenner_Schlumberger.npy"),
    model)

# Maillage FEM
mesh.save(
    os.path.join(output_dir, "maillage_FEM_Wenner_Schlumberger.bms"))

# Données ERT
data.save(
    os.path.join(output_dir, "donnees_synthetiques_Wenner_Schlumberger.dat"))

# Sauvegarde complète pyGIMLi
mgr.saveResult(
    folder=output_dir)

print("\n================================")
print("SIMULATION TERMINEE")
print("Fichiers sauvegardés :")
print("--------------------------------")
print(f"- {os.path.join(fig_dir, '01_modele_geologique.png')}")
print(f"- {os.path.join(fig_dir, '02_maillage_FEM.png')}")
print(f"- {os.path.join(fig_dir, '03_pseudosection_synthetique.png')}")
print(f"- {os.path.join(fig_dir, '04_resultat_inversion.png')}")
print(f"- {os.path.join(fig_dir, '05_schema_global.png')}")
print(f"- {os.path.join(output_dir, 'modele_inverse_Wenner_Schlumberger.npy')}")
print(f"- {os.path.join(output_dir, 'maillage_FEM_Wenner_Schlumberger.bms')}")
print(f"- {os.path.join(output_dir, 'donnees_synthetiques_Wenner_Schlumberger.dat')}")
print(f"- dossier {output_dir}/")
print("================================")